# Redis Vector DB with Docker: End-to-End Demo

This notebook demonstrates how to inspect and interact with a Redis vector database using a Docker-based Redis instance. It covers setup, data loading, index creation, and advanced search queries, following the latest RedisVL Query API best practices.

## Install and Import Required Libraries

Install the necessary Python packages for working with Redis, vector search, and data processing. Then, import them for use in this notebook.

In [1]:
# Uncomment and run if needed
# !pip install redis redis-py-cluster redisvl pandas numpy openai

import redis
import numpy as np
import pandas as pd
import os
import openai
# from redisvl import Query  # For RedisVL Query API (if using redisvl)


## Configure Redis Connection for Docker Instance

Set the Redis host, port, and password to match your Docker container settings. By default, Docker exposes Redis on `localhost` and port `6379` unless mapped differently.

In [2]:
# Set your Docker Redis connection details
REDIS_HOST = 'localhost'  # or your Docker host IP
REDIS_PORT = 6379         # default Redis port, change if mapped differently
REDIS_PASSWORD = None     # set if your Redis instance requires a password

## Connect to Redis (Docker) and Verify

Connect to the Redis instance running in Docker and verify the connection using `ping` and `info` commands.

In [5]:
# Connect to Redis Docker instance and verify
try:
    r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, password=REDIS_PASSWORD, decode_responses=True)
    assert r.ping(), "Could not connect to Redis!"
    print('Connected to Redis (Docker)!')
    info = r.info()
    print(f"Redis version: {info.get('redis_version')}")
    print(f"Connected clients: {info.get('connected_clients')}")
    print(f"Used memory: {info.get('used_memory_human')}")
    print(f"Total keys: {r.dbsize()}")
except Exception as e:
    print(f'Error connecting to Redis: {e}')

Connected to Redis (Docker)!
Redis version: 7.4.7
Connected clients: 1
Used memory: 1.32M
Total keys: 0


## Load and Inspect Sample Data

Load a sample CSV or dataset, display its structure, and check vector and metadata columns. (Replace the file path with your actual dataset location if needed.)

In [6]:
# Load sample data (update the path as needed)
#data_path = r"D:\AI-DATASETS\02-MISC-large\GenAI-LLMs\vector_database_wikipedia_articles_embedded_100.csv"
data_path = r"D:\Makesh\Working\AI\RPS\Day09\vector_database_wikipedia_articles_embedded.csv"
article_df = pd.read_csv(data_path)
article_df.head()

,id,url,title,text,title_vector,content_vector,vector_id
0,1,https://simple.wikipedia.org/wiki/April,April,April is the fourth month of the year in the J...,"[0.001009464613161981, -0.020700545981526375, ...","[-0.011253940872848034, -0.013491976074874401,...",0
1,2,https://simple.wikipedia.org/wiki/August,August,August (Aug.) is the eighth month of the year ...,"[0.0009286514250561595, 0.000820168002974242, ...","[0.0003609954728744924, 0.007262262050062418, ...",1
2,6,https://simple.wikipedia.org/wiki/Art,Art,Art is a creative activity that expresses imag...,"[0.003393713850528002, 0.0061537534929811954, ...","[-0.004959689453244209, 0.015772193670272827, ...",2
3,8,https://simple.wikipedia.org/wiki/A,A,A or a is the first letter of the English alph...,"[0.0153952119871974, -0.013759135268628597, 0....","[0.024894846603274345, -0.022186409682035446, ...",3
4,9,https://simple.wikipedia.org/wiki/Air,Air,Air refers to the Earth's atmosphere. Air is a...,"[0.02224554680287838, -0.02044147066771984, -0...","[0.021524671465158463, 0.018522677943110466, -...",4


In [7]:
article_df.shape

(25000, 7)

In [8]:
article_df.columns

Index(['id', 'url', 'title', 'text', 'title_vector', 'content_vector',
       'vector_id'],
      dtype='object')

In [9]:
# Inspect the dataframe structure
article_df.info(show_counts=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              25000 non-null  int64 
 1   url             25000 non-null  object
 2   title           25000 non-null  object
 3   text            25000 non-null  object
 4   title_vector    25000 non-null  object
 5   content_vector  25000 non-null  object
 6   vector_id       25000 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 1.3+ MB


## Prepare Vectors and Metadata

Convert vector columns from string to list/array, ensure IDs are strings, and preprocess data for Redis ingestion.

In [10]:
%%time

from ast import literal_eval

# Convert vector columns from string to list
article_df['title_vector']   = article_df['title_vector'].apply(literal_eval)
article_df['content_vector'] = article_df['content_vector'].apply(literal_eval)

# Ensure vector_id is a string
article_df['vector_id'] = article_df['vector_id'].apply(str)

article_df.head()

CPU times: total: 8min 27s
Wall time: 9min 10s


,id,url,title,text,title_vector,content_vector,vector_id
0,1,https://simple.wikipedia.org/wiki/April,April,April is the fourth month of the year in the J...,"[0.001009464613161981, -0.020700545981526375, ...","[-0.011253940872848034, -0.013491976074874401,...",0
1,2,https://simple.wikipedia.org/wiki/August,August,August (Aug.) is the eighth month of the year ...,"[0.0009286514250561595, 0.000820168002974242, ...","[0.0003609954728744924, 0.007262262050062418, ...",1
2,6,https://simple.wikipedia.org/wiki/Art,Art,Art is a creative activity that expresses imag...,"[0.003393713850528002, 0.0061537534929811954, ...","[-0.004959689453244209, 0.015772193670272827, ...",2
3,8,https://simple.wikipedia.org/wiki/A,A,A or a is the first letter of the English alph...,"[0.0153952119871974, -0.013759135268628597, 0....","[0.024894846603274345, -0.022186409682035446, ...",3
4,9,https://simple.wikipedia.org/wiki/Air,Air,Air refers to the Earth's atmosphere. Air is a...,"[0.02224554680287838, -0.02044147066771984, -0...","[0.021524671465158463, 0.018522677943110466, -...",4


## Define and Create RediSearch Index (Docker)

Define the RediSearch schema (TextField, VectorField, etc.) and create the index in the Docker Redis instance.

In [12]:
from redis.commands.search.field import TextField, VectorField
from redis.commands.search.index_definition import IndexDefinition, IndexType

# Define schema fields for the Wikipedia CSV
fields = [
    TextField("id"),
    TextField("url"),
    TextField("title"),
    TextField("text"),
    VectorField("title_vector", "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": len(article_df['title_vector'][0]),
        "DISTANCE_METRIC": "COSINE",
        "INITIAL_CAP": len(article_df)
    }),
    VectorField("content_vector", "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": len(article_df['content_vector'][0]),
        "DISTANCE_METRIC": "COSINE",
        "INITIAL_CAP": len(article_df)
    }),
    TextField("vector_id")
]

INDEX_NAME = "wikipedia-index"
PREFIX = "wiki:"

# Create the index if it doesn't exist
# RediSearch Index: This command creates a RediSearch index in Redis, allowing you to run fast full-text, vector, 
# and hybrid queries.

# index_type=IndexType.HASH: This tells RediSearch to index documents stored as Redis Hashes 
# (the default Redis data structure for storing field-value pairs).

# prefix=[PREFIX]: Only keys starting with this prefix (e.g., "wiki:") will be indexed. 
# This helps organize and isolate your indexed data.

# fields=fields: Specifies which fields in each Hash will be indexed (e.g., text, vectors).

# How it works in practice:

# You store each document as a Redis Hash (e.g., wiki:1234), with fields like title, text, title_vector, etc.
# RediSearch automatically indexes these fields for all Hashes with the given prefix.
# You can then run fast search queries (text, vector, hybrid) over these indexed fields.
try:
    r.ft(INDEX_NAME).info()
    print("Index already exists.")
except:
    r.ft(INDEX_NAME).create_index(
        fields      = fields,
        definition  = IndexDefinition(prefix=[PREFIX], index_type=IndexType.HASH)
    )
    print("Index created.")

Index already exists.


In [ ]:
# Load data into the index
# takes about 5-7 minutes for 25K records

def index_documents(client, prefix, df):
    # Convert DataFrame to list of dictionaries where each dictionary represents a document/row
    # [
    #   {"id": 1, "title": "Hello", "title_vector": [0.1, 0.2], "content_vector": [0.5, 0.6]},
    #   {"id": 2, "title": "World", "title_vector": [0.3, 0.4], "content_vector": [0.7, 0.8]}
    # ]
    records = df.to_dict("records")
    for doc in records:
        key = f"{prefix}{doc['id']}"
        # Convert vectors to bytes
        doc["title_vector"] = np.array(doc["title_vector"], dtype=np.float32).tobytes()
        doc["content_vector"] = np.array(doc["content_vector"], dtype=np.float32).tobytes()
        client.hset(key, mapping=doc)

index_documents(r, PREFIX, article_df)
print(f"Loaded {r.dbsize()} documents into Redis index '{INDEX_NAME}' with prefix '{PREFIX}'")

Loaded 25000 documents into Redis index 'wikipedia-index' with prefix 'wiki:'
